In [ ]:
pip install findspark

In [1]:
import socket

server_host = 'HYPE' # Or the IP address of HYPE
server_port = 1433

try:
    with socket.create_connection((server_host, server_port), timeout=5):
        print(f"Successfully connected to {server_host} on port {server_port}.")
except socket.error as e:
    print(f"Connection failed: {e}")
    print("Please check the SQL Server configuration and firewall settings on the server machine.")


Successfully connected to HYPE on port 1433.


In [2]:
import os
import findspark

# --- Set the SPARK_HOME environment variable manually ---
# Use forward slashes or double backslashes in the path
os.environ["SPARK_HOME"] = "C:/spark" 

findspark.init()
print("Findspark initialized successfully.") # Added confirmation message

Findspark initialized successfully.


In [3]:
import os
import findspark
from pyspark.sql import SparkSession
import pyspark

# --- Set the SPARK_HOME environment variable manually ---
# Use forward slashes or double backslashes in the path
os.environ["SPARK_HOME"] = "C:/spark" 

findspark.init()
print("Findspark initialized successfully.") 

# Define the paths to your local JAR files, now with the file:/// scheme
# Use forward slashes exclusively in the final path string
mssql_jar_path = r"file:///C:/spark_drivers/mssql-jdbc-13.2.1.jre11.jar"
mysql_jar_path = r"file:///C:/spark_drivers/mysql-connector-j-9.5.0.jar"

spark = SparkSession.builder \
    .appName("SQLServerToMySQLTransfer") \
    .config("spark.jars", f"{mssql_jar_path},{mysql_jar_path}") \
    .getOrCreate()

print("Spark session created.")

# You can now proceed with your data transfer logic here
# For example:
# df = spark.read.format("jdbc").option(...)

Findspark initialized successfully.
Spark session created.


In [6]:
import os
import findspark
import time
from pyspark.sql import SparkSession
import pyspark

# --- Initialization (Assuming this part is already successful) ---
os.environ["SPARK_HOME"] = "C:/spark" 
findspark.init()

mssql_jar_path = r"file:///C:/spark_drivers/mssql-jdbc-13.2.1.jre11.jar"
mysql_jar_path = r"file:///C:/spark_drivers/mysql-connector-j-9.5.0.jar"

spark = SparkSession.builder \
    .appName("SQLServerToMySQLTransfer") \
    .config("spark.jars", f"{mssql_jar_path},{mysql_jar_path}") \
    .getOrCreate()

print("Spark session created successfully.")

# --- Data Transfer and Timing ---

# Start the timer
start_time = time.time()

# Connection details for SQL Server (Source)
# Note the added encrypt/trustServerCertificate properties as previously solved
sql_server_url = r"jdbc:sqlserver://HYPE\SQL_2022:1433;databaseName=sttwk_2025a_par;encrypt=true;trustServerCertificate=true"
sql_server_user = "sa"
sql_server_password = "D@1ry$2014"
sql_server_table = "sttwk"

# Read data into a PySpark DataFrame
df = spark.read.format("jdbc").options(
    url=sql_server_url,
    dbtable=sql_server_table,
    user=sql_server_user,
    password=sql_server_password,
    driver="com.microsoft.sqlserver.jdbc.SQLServerDriver"
).load()

source_count = df.count()
print(f"Number of rows read from SQL Server: {source_count}")

# Connection details for MySQL (Destination)
mysql_url = "jdbc:mysql://localhost:3306/AdventureSales"
mysql_user = "root"
mysql_password = "" # Blank password for XAMPP default
mysql_table = "sttwk"

# Write the DataFrame to the MySQL table
df.write.format("jdbc").options(
    url=mysql_url,
    dbtable=mysql_table,
    user=mysql_user,
    password=mysql_password,
    driver="com.mysql.cj.jdbc.Driver"
).mode("overwrite").save()

# Stop the timer
end_time = time.time()

# Calculate and print the total time taken
total_time = end_time - start_time
print(f"Data written to MySQL successfully.")
print(f"Total time taken for transfer: {total_time:.2f} seconds")


# --- Optional Verification ---

df_verify = spark.read.format("jdbc").options(
    url=mysql_url,
    dbtable=mysql_table,
    user=mysql_user,
    password=mysql_password,
    driver="com.mysql.cj.jdbc.Driver"
).load()

mysql_count = df_verify.count()
print(f"Verified count in MySQL destination table: {mysql_count}")

# Stop the Spark session (optional but good practice)
spark.stop()





Spark session created successfully.
Number of rows read from SQL Server: 88075099
Data written to MySQL successfully.
Total time taken for transfer: 10446.34 seconds


ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "C:\spark\python\pyspark\errors\exceptions\captured.py", line 282, in deco
    return f(*a, **kw)
  File "C:\spark\python\lib\py4j-0.10.9.9-src.zip\py4j\protocol.py", line 327, in get_return_value
    raise Py4JJavaError(
        "An error occurred while calling {0}{1}{2}.\n".
        format(target_id, ".", name), value)
py4j.protocol.Py4JJavaError: <exception str() failed>

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "C:\spark\python\lib\py4j-0.10.9.9-src.zip\py4j\clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ~~~~~~~~~~~~~~~~~~~~^^
  File "C:\Python313\Lib\socket.py", line 719, in readinto
    return self._sock.recv_into(b)
           ~~~~~~~~~~~~~~~~~~~~^^^
ConnectionResetError: [WinError 10054] An existing connection was forcibly closed by the remote host

Py4JError: py4j does not exist in the JVM

In [5]:
# Assuming 'spark' session is still active from Cell 1

# Connection details for MySQL
mysql_url = "jdbc:mysql://localhost:3306/AdventureSales"
mysql_user = "root"
mysql_password = "" 

# Define a custom query to read only the top 5 records
# This pushes the LIMIT clause down to the MySQL server itself
mysql_query = "(SELECT count(*) FROM sttwk) AS data_count"

# Read a limited selection of data into a PySpark DataFrame
df_sample = spark.read.format("jdbc").options(
    url=mysql_url,
    dbtable=mysql_query,
    user=mysql_user,
    password=mysql_password,
    driver="com.mysql.cj.jdbc.Driver"
).load()

print("Displaying count of  records from MySQL table 'sttwk':")
# Use .show() to display the records
df_sample.show()

# To get the count of the sample:
print(f"Number of sample rows read: {df_sample.count()}")


Displaying count of  records from MySQL table 'sttwk':
+--------+
|count(*)|
+--------+
|88075099|
+--------+

Number of sample rows read: 1
